# Práctica 3 · Inferencia e intro a Machine Learning

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Lautaromgo/estadistica-aplicada-unne/blob/main/notebooks/practica/practica_3_inferencia_ml.ipynb)

Dos partes, las dos de la clase 3:

- **Parte A · Inferencia**, con los números reales del informe de pobreza del INDEC
  (primer semestre de 2026). No hay dataset: los datos vienen cargados en el setup.
- **Parte B · Machine Learning**, con los 5.000 préstamos **inventados** de la notebook 06.

Funciona igual que las anteriores: seis funciones vacías con un `TODO` adentro, y una
celda de verificación después de cada una. Mientras no completes una función, las celdas
que la usan muestran `NotImplementedError`. Es esperable: no rompiste nada.

Corré todo de arriba hacia abajo.

## 0 · Setup

Trae los datos del INDEC, el dataset de créditos, `verificar()` y `p_valor()`, que ya
viene hecho.

In [ ]:
import os
from math import erf, sqrt

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

sns.set_theme(style="whitegrid")
pd.set_option("display.width", 140)

# --- Parte A: INDEC, EPH, 1er semestre 2026 (cuadros 4.1 y 4.2) -------------
# Incidencia en PERSONAS (%) y su coeficiente de variación (%).
indec = pd.DataFrame({
    "area": ["Total 31 aglomerados", "Noreste", "Corrientes", "Formosa",
             "Gran Resistencia", "Posadas"],
    "pobreza":        [32.3, 41.5, 41.0, 29.9, 48.6, 42.2],
    "cv_pobreza":     [ 2.7,  5.8, 10.4,  9.7, 10.8, 12.5],
    "indigencia":     [ 7.5, 10.5,  8.6,  5.9, 17.3,  8.2],
    "cv_indigencia":  [ 7.4, 13.7, 26.9, 41.9, 21.6, 29.9],
}).set_index("area")

# --- Parte B: los préstamos inventados de la notebook 06 --------------------
URL = ("https://raw.githubusercontent.com/Lautaromgo/estadistica-aplicada-unne/"
       "main/data/raw/creditos_sinteticos.csv")
RUTAS = ["../../data/raw/creditos_sinteticos.csv", "../data/raw/creditos_sinteticos.csv",
         "creditos_sinteticos.csv"]
ruta = next((r for r in RUTAS if os.path.exists(r)), None)
creditos = pd.read_csv(ruta) if ruta else pd.read_csv(URL)
creditos["cuota_sobre_ingreso"] = creditos["cuota"] / creditos["ingreso_mensual"]


def verificar(nombre, obtenido, esperado, tol=1e-6):
    """Compara tu resultado con el esperado. Avisa y sigue: no corta la notebook."""
    try:
        if isinstance(esperado, str):
            ok = obtenido == esperado
        else:
            ok = bool(np.isclose(float(obtenido), float(esperado), rtol=tol, atol=tol))
    except (TypeError, ValueError):
        ok = False
    print(f"✅ {nombre}" if ok
          else f"❌ {nombre} → obtuviste {obtenido!r}, se esperaba {esperado!r}")


def nro(x, decimales=0):
    """Formatea un número como lo escribimos nosotros: 20.640 · 0,98"""
    return f"{x:,.{decimales}f}".replace(",", "|").replace(".", ",").replace("|", ".")


def p_valor(z):
    """Probabilidad de quedar a |z| errores estándar o más de H0, por puro azar."""
    return 2 * (1 - 0.5 * (1 + erf(abs(z) / sqrt(2))))


print(f"INDEC: {len(indec)} áreas   |   créditos: {nro(len(creditos))} préstamos "
      f"(origen: {ruta or 'la web'})")

# Parte A · Inferencia con los números del INDEC

## 1 · Del CV al error estándar

El INDEC no publica el error estándar de cada aglomerado: publica el **coeficiente de
variación**. Pero el CV ya lo conocés del punto 2: es el desvío dividido por la media,
en porcentaje. Acá el "desvío" es el de la **estimación**, o sea el error estándar:

```
CV = EE / estimación × 100      →      EE = estimación × CV / 100
```

In [ ]:
def error_estandar(estimacion, cv):
    """El error estándar de una estimación, a partir de su CV (los dos en %).

    Devuelve el EE en puntos porcentuales. Por ejemplo, 32,3% con CV 2,7% da un EE de
    0,87 puntos.
    """
    # ─── TODO ────────────────────────────────────────────────────────────
    #   1. Despejá EE de la fórmula de arriba: estimación × CV / 100.
    #   2. Devolvelo.
    # ─────────────────────────────────────────────────────────────────────
    raise NotImplementedError("Falta completar error_estandar(). Mirá el TODO justo arriba.")

In [ ]:
verificar("error_estandar · total del país", error_estandar(32.3, 2.7), 0.8721)
verificar("error_estandar · Corrientes", error_estandar(41.0, 10.4), 4.264)

## 2 · El intervalo de confianza

La correa del perro: `estimación ± Z × EE`. El INDEC usa **90%**, así que Z = 1,645.

In [ ]:
def intervalo(estimacion, ee, z=1.645):
    """Intervalo de confianza: (inferior, superior) = estimación ± z × EE.

    Por defecto z = 1,645 (90%), como el INDEC. Para 95% se pasa z=1.96.
    """
    # ─── TODO ────────────────────────────────────────────────────────────
    #   1. Calculá el margen de error: z × ee.
    #   2. Devolvé una tupla (estimacion - margen, estimacion + margen).
    # ─────────────────────────────────────────────────────────────────────
    raise NotImplementedError("Falta completar intervalo(). Mirá el TODO justo arriba.")

In [ ]:
inf, sup = intervalo(32.3, error_estandar(32.3, 2.7))
verificar("intervalo · límite inferior", inf, 32.3 - 1.645 * 0.8721)
verificar("intervalo · límite superior", sup, 32.3 + 1.645 * 0.8721)
print(f"\nTu intervalo: [{nro(inf, 1)} ; {nro(sup, 1)}]   ·   el INDEC publica [30,8 ; 33,7]")

#### ⚠️ Probalo y mirá (1): el mismo método, anchos muy distintos

In [ ]:
filas = []
for area, fila in indec.iterrows():
    ee = error_estandar(fila["pobreza"], fila["cv_pobreza"])
    inf, sup = intervalo(fila["pobreza"], ee)
    filas.append({"área": area, "pobreza %": fila["pobreza"],
                  "IC 90%": f"[{nro(inf, 1)} ; {nro(sup, 1)}]", "ancho (pp)": round(sup - inf, 1)})
print(pd.DataFrame(filas).set_index("área").to_string())

El intervalo de Posadas es **6 veces más ancho** que el del total del país. El
método es el mismo; lo que cambia es cuántas viviendas se encuestaron en cada lugar.

## 3 · El semáforo del CV

La regla del INDEC (anexo metodológico del informe):

| CV | lectura |
|---|---|
| menos de 16% | `"confiable"` |
| de 16% a 25% | `"con cuidado"` |
| más de 25% | `"no confiable"` |

In [ ]:
def semaforo(cv):
    """Devuelve "confiable", "con cuidado" o "no confiable" según el CV (en %)."""
    # ─── TODO ────────────────────────────────────────────────────────────
    #   1. Si cv es menor que 16, devolvé "confiable".
    #   2. Si no, y cv es 25 o menos, devolvé "con cuidado".
    #   3. Si no, devolvé "no confiable".
    # ─────────────────────────────────────────────────────────────────────
    raise NotImplementedError("Falta completar semaforo(). Mirá el TODO justo arriba.")

In [ ]:
verificar("semaforo · 2,7", semaforo(2.7), "confiable")
verificar("semaforo · 16", semaforo(16), "con cuidado")
verificar("semaforo · 21,6", semaforo(21.6), "con cuidado")
verificar("semaforo · 26,9", semaforo(26.9), "no confiable")

#### ⚠️ Probalo y mirá (2): la indigencia en el NEA

In [ ]:
tabla = indec[["indigencia", "cv_indigencia"]].copy()
tabla["semáforo"] = tabla["cv_indigencia"].apply(semaforo)
print(tabla.to_string())

La pobreza de Corrientes se puede usar tranquilo (CV 10,4%). **La indigencia de
Corrientes, no** (CV 26,9%): es el `ISLAND` de la vida real. Mismo aglomerado, misma
encuesta, dos conclusiones distintas: el n chico no invalida todo por igual.

## 4 · El contraste: ¿la diferencia es real?

Los tres pasos de la clase:

1. Suponemos H0: "no hay diferencia". La diferencia debería rondar el 0, con una campana
   de ancho `EE_dif = √(EE_a² + EE_b²)`.
2. `z = diferencia / EE_dif`: a cuántos errores estándar de "no hay diferencia" quedó lo
   observado.
3. `p_valor(z)` (ya está hecha en el setup) dice qué tan raro sería eso si H0 fuera cierta.

> Esta cuenta supone que las dos estimaciones son independientes. Para aglomerados
> distintos de la misma encuesta es una aproximación razonable.

In [ ]:
def contraste(est_a, ee_a, est_b, ee_b):
    """Contraste de la diferencia A − B. Devuelve un dict con dif, ee_dif, z y p."""
    # ─── TODO ────────────────────────────────────────────────────────────
    #   1. dif = est_a - est_b
    #   2. ee_dif = raíz cuadrada de (ee_a² + ee_b²). Podés usar np.sqrt().
    #   3. z = dif / ee_dif
    #   4. p = p_valor(z)
    #   5. Devolvé {"dif": dif, "ee_dif": ee_dif, "z": z, "p": p}
    # ─────────────────────────────────────────────────────────────────────
    raise NotImplementedError("Falta completar contraste(). Mirá el TODO justo arriba.")

In [ ]:
# El caso de la clase: ¿Gran Resistencia es más pobre que Corrientes?
r = contraste(48.6, error_estandar(48.6, 10.8), 41.0, error_estandar(41.0, 10.4))
verificar("contraste · diferencia", r["dif"], 7.6)
verificar("contraste · z", r["z"], 7.6 / np.sqrt(5.2488 ** 2 + 4.264 ** 2))
verificar("contraste · p", r["p"], p_valor(7.6 / np.sqrt(5.2488 ** 2 + 4.264 ** 2)))
print(f"\nz = {nro(r['z'], 2)}   p = {nro(r['p'], 2)}   → falta de pruebas: no se rechaza H0")

#### ⚠️ Probalo y mirá (3): Formosa contra Posadas

Formosa tiene 29,9% de pobreza y Posadas 42,2%. Completá la cuenta y decidí.

In [ ]:
fo, po = indec.loc["Formosa"], indec.loc["Posadas"]
r = contraste(fo["pobreza"], error_estandar(fo["pobreza"], fo["cv_pobreza"]),
              po["pobreza"], error_estandar(po["pobreza"], po["cv_pobreza"]))

print(f"diferencia = {nro(r['dif'], 1)} pp   EE_dif = {nro(r['ee_dif'], 2)}")
print(f"z = {nro(r['z'], 2)}   p = {nro(r['p'], 3)}")
for nivel, z_critico in [("90%", 1.645), ("95%", 1.96)]:
    decision = "se rechaza H0" if abs(r["z"]) > z_critico else "no se rechaza H0"
    print(f"al {nivel}: {decision}")

# Parte B · Machine Learning con los préstamos inventados

## 5 · ¿Cuánto acierta? Y contra qué

Antes de medir cualquier modelo hace falta la vara: el **baseline**. Primero, una función
para contar aciertos.

In [ ]:
def exactitud(y_real, y_pred):
    """Proporción de casos en que la predicción coincide con lo que pasó (entre 0 y 1)."""
    # ─── TODO ────────────────────────────────────────────────────────────
    #   1. Pasá las dos cosas a arrays: np.asarray(...).
    #   2. Compará elemento a elemento con ==. Eso da un array de True/False.
    #   3. Devolvé la media de ese array (True cuenta como 1, False como 0).
    # ─────────────────────────────────────────────────────────────────────
    raise NotImplementedError("Falta completar exactitud(). Mirá el TODO justo arriba.")

In [ ]:
y = creditos["pago"]
verificar("exactitud", exactitud([1, 0, 1, 1], [1, 1, 1, 0]), 0.5)

aprobar_a_todos = np.ones(len(y), dtype=int)
verificar("exactitud del baseline", exactitud(y, aprobar_a_todos), y.mean())
print(f"\nAprobar a todos, sin mirar nada, acierta el {nro(100 * exactitud(y, aprobar_a_todos), 1)}%.")

El modelo de la notebook 06, ya entrenado. No tenés que tocar nada: es para tener un
score con el que trabajar.

In [ ]:
FEATURES = ["edad", "ingreso_mensual", "cuota", "cuota_sobre_ingreso",
            "antiguedad_laboral", "atrasos_12m", "consultas_recientes", "dia_solicitud"]
X = pd.get_dummies(creditos[FEATURES], columns=["dia_solicitud"], dtype=int)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y)

arbol = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X_train, y_train)
score = (arbol.predict_proba(X_test)[:, 1] * 1000).round()

print(f"baseline en test:          {nro(100 * exactitud(y_test, np.ones(len(y_test))), 1)}%")
print(f"árbol de 3 preguntas:      {nro(100 * exactitud(y_test, arbol.predict(X_test)), 1)}%")

## 6 · El umbral: quién entra y quién no

La financiera aprueba a quien tenga `score >= umbral`. Cada umbral reparte los errores
de otra manera:

- **buenos rechazados:** no se aprobaron, pero habrían pagado (`pago == 1`);
- **morosos aprobados:** se aprobaron y no pagaron (`pago == 0`).

In [ ]:
def evaluar_umbral(score, y_real, umbral):
    """Cuántos se aprueban y cuántos errores de cada tipo hay con un umbral dado.

    Devuelve un dict:
        "aprobados"          proporción de aprobados (entre 0 y 1)
        "morosos_aprobados"  cantidad de aprobados que NO pagaron
        "buenos_rechazados"  cantidad de rechazados que SÍ habrían pagado
    """
    # ─── TODO ────────────────────────────────────────────────────────────
    #   1. Pasá score e y_real a arrays con np.asarray().
    #   2. aprueba = score >= umbral        (array de True/False)
    #   3. aprobados = la media de aprueba
    #   4. morosos_aprobados = cuántos cumplen aprueba Y y_real == 0
    #      (combiná condiciones con &, y contá con .sum())
    #   5. buenos_rechazados = cuántos cumplen NO aprueba (~aprueba) Y y_real == 1
    #   6. Devolvé el dict con esas tres claves.
    # ─────────────────────────────────────────────────────────────────────
    raise NotImplementedError("Falta completar evaluar_umbral(). Mirá el TODO justo arriba.")

In [ ]:
r = evaluar_umbral([900, 400, 700, 100], [1, 1, 0, 0], 500)
verificar("evaluar_umbral · aprobados", r["aprobados"], 0.5)
verificar("evaluar_umbral · morosos aprobados", r["morosos_aprobados"], 1)
verificar("evaluar_umbral · buenos rechazados", r["buenos_rechazados"], 1)

r = evaluar_umbral(score, y_test, 500)
verificar("evaluar_umbral · con el árbol", r["morosos_aprobados"],
          int(((score >= 500) & (y_test.values == 0)).sum()))

#### ⚠️ Probalo y mirá (4): elegí vos el umbral

Supongamos que un moroso le cuesta a la financiera **10 veces** lo que le cuesta perder a
un buen cliente. El "costo" de cada umbral sería:

```
costo = 10 × morosos aprobados + 1 × buenos rechazados
```

In [ ]:
COSTO_MOROSO, COSTO_BUENO = 10, 1

filas = []
for umbral in [0, 300, 500, 700, 950]:
    r = evaluar_umbral(score, y_test, umbral)
    filas.append({"umbral": umbral, "aprobados": f"{r['aprobados']:.0%}",
                  "morosos aprobados": r["morosos_aprobados"],
                  "buenos rechazados": r["buenos_rechazados"],
                  "costo": COSTO_MOROSO * r["morosos_aprobados"] + COSTO_BUENO * r["buenos_rechazados"]})
print(pd.DataFrame(filas).set_index("umbral").to_string())

Cambiá `COSTO_MOROSO` a 2 y volvé a correr la celda. **El mejor umbral cambia.** El modelo
es el mismo; lo que cambió es cuánto te duele cada error, y eso no lo decide la
estadística.

## ✍️ Para pensar

1. **El INDEC informa que la indigencia en Corrientes es 8,6%.** Un diario titula
   *"Corrientes tiene menos indigencia que el promedio del NEA (10,5%)"*. Con lo que
   hiciste en las secciones 1 a 4, escribí en dos líneas por qué ese titular no se
   sostiene.

2. **El árbol de la parte B se entrenó con los préstamos que la financiera aprobó.**
   ¿Qué pasa con la gente que siempre fue rechazada? Relacionalo con la sopa de la
   clase: ¿de qué parte de la olla está sacando la cucharada el modelo?

*Escribí tus respuestas acá (doble clic para editar esta celda).*

**1.**

**2.**